In [ ]:
# pip install pytesseract pillow gspread oauth2client

import pytesseract
from PIL import Image
import gspread
from oauth2client.service_account import ServiceAccountCredentials
import re

# === CONFIGURAÇÕES ===
import os

caminho_imagem = os.getenv("IMAGEM_EXAME", "exemplos/exame.jpg")
chave_google_json = os.environ["GOOGLE_APPLICATION_CREDENTIALS"]
nome_planilha = os.getenv("PLANILHA_DESTINO", "bioimpedancia")
nome_aba = "Dados"

# Se precisar, configure o caminho do Tesseract no Windows:
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'



In [ ]:
# === OCR ===
imagem = Image.open(caminho_imagem)
texto = pytesseract.image_to_string(imagem, lang='por')

# === EXTRAÇÃO DE CAMPOS (exemplo simplificado com regex) ===
def extrair_valor(titulo, texto):
    padrao = re.compile(rf"{titulo}.*?([\d,]+)", re.IGNORECASE)
    match = padrao.search(texto)
    return match.group(1).replace(",", ".") if match else ""

dados = {
    "Data": extrair_valor("Data / Hora", texto),
    "Peso": extrair_valor("Peso \\(kg\\)", texto),
    "Água Corporal Total": extrair_valor("Água Corporal Total (L) \\(\\)", texto),
    "Proteína": extrair_valor("Proteína", texto),
    "Minerais": extrair_valor("Minerais", texto),
    "Massa de Gordura": extrair_valor("Massa de Gordura", texto),
    "Massa Muscular Esquelética": extrair_valor("Massa Muscular Esquelética", texto),
    "Taxa Metabólica Basal": extrair_valor("Taxa Metabólica Basal", texto),
    "IMC": extrair_valor("IMC", texto),
    "PGC": extrair_valor("PGC", texto),
    "Pontuação InBody": extrair_valor("Pontuação InBody", texto)
}
"""
##  Ver o texto bruto extraído
print("🧾 Texto bruto OCR extraído:")
print(texto)
"""

print("📋 Dados extraídos:")
for k, v in dados.items():
    print(f"{k}: {v}")

In [ ]:
import re

# Cole aqui o texto OCR bruto que você forneceu
texto_ocr = texto


def extrair_com_regex(padrao, texto):
    """ Tenta encontrar o padrão no texto e retorna o grupo 1. """
    # re.DOTALL faz com que '.' também corresponda a quebras de linha
    # re.IGNORECASE ignora maiúsculas/minúsculas
    match = re.search(padrao, texto, re.IGNORECASE | re.DOTALL)
    if match:
        return match.group(1).strip()
    return None # Retorna None se não encontrar

# === PADRÕES DE REGEX (AJUSTADOS PARA O SEU OCR) ===
padroes = {
    # Data: Procura 'Data / Hora', pula a linha e captura o padrão DD.MM.YYYY. HH:MM
    #"Data": r"Data / Hora\s*([\d\.\s:]+?)\s*\n",
    "Data": r"Data / Hora\s*(\d{2}\.\d{2}\.\d{4})",
    # Peso: Procura 'Peso ) —', ignora um 'G' opcional e captura o número com vírgula.
    "Peso": r"Peso\s*\)\s*—\s*G?([\d,\.]+)",
    # Água: Procura a etiqueta exata e captura o número.
    "Água Corporal Total": r"Água Corporal Total \(L\)\s*([\d,\.]+)",
    # Proteína: Procura a etiqueta e captura o número (que vem como 1106).
    #"Proteína": r"Proteína \(kg\)\s*([\d,\.]+)",
    "Proteína": r"Proteína \(kg\)\s*([\d,\.]+)",
    # Minerais: Procura a etiqueta e captura o número.
    "Minerais": r"Minerais \(kg\)\s*([\d,\.]+)",
    # Massa Gordura: Padrão que PROVAVELMENTE FALHARÁ, pois o OCR é ruim aqui.
    "Massa de Gordura": r"Massa de Gordura \(kg\)\s*.*?([\d,\.]+)",
    # Massa Muscular: Usa 'Esquelética' como âncora e pega o número.
    "Massa Muscular Esquelética": r"Massa Muscular Esqueléica = \s*[-_l\s]*([\d,\.]+)",
    # TMB: Procura a etiqueta (sem espaço) e pega o número.
    "Taxa Metabólica Basal": r"TaxaMetabóicaBasal\s*([\d]+)",
    # IMC: Tenta pegar o número perto de 'Índice de Massa Corporal'. MUITO INCERTO.
    "IMC": r"Índice de Massa Comporal\s*.*?([\d,\.]+)",
    # PGC: Tenta pegar o número perto de 'Porcentual de Gordura'. MUITO INCERTO.
    #"PGC": r"'PGC'\s*[\s]*?([\d,\.]+)",
    "PGC": r"PGC\s*(?:&\) \|)?\s*([\d,\.]+)",
    # Pontuação: Procura a etiqueta e pega o número antes da barra.
    "Pontuação InBody": r"Pontuação InBody\s*([\d]+)\s*/",
}

# === EXTRAÇÃO E LIMPEZA ===
dados_extraidos = {}
print("📋 Dados extraídos:")

for campo, padrao in padroes.items():
    valor = extrair_com_regex(padrao, texto_ocr)
    
    if valor:
        if campo == "Data":
            # Limpa o ponto extra da data
            valor_limpo = re.sub(r'(\d{4})\.\s', r'\1 ', valor)
        elif campo == "IMC":
            # Tenta limpar o ", 7" -> ".7" (Exemplo, pode precisar de mais lógica)
            valor_limpo = valor.replace(",", ".").replace(" ", "")
      
        elif campo == "Proteína":
             valor_limpo = float(valor[:3])/10
         
        else:
            # Para outros campos, só troca vírgula por ponto
            valor_limpo = valor

        dados_extraidos[campo] = valor_limpo
        print(f"{campo}: {valor_limpo}")
    else:
        dados_extraidos[campo] = "Não encontrado"
        print(f"{campo}: Não encontrado")

In [ ]:
from openpyxl import Workbook, load_workbook
from openpyxl.utils import get_column_letter
caminho_excel = os.getenv("EXCEL_DESTINO", "dados_bioimpedancia.xlsx")
nome_aba_excel = "Dados InBody"
dados = dados_extraidos

def salvar_no_excel(dados, caminho_excel, nome_aba_excel):
    """
    Salva os dados extraídos em um arquivo Excel local.
    Cria o arquivo e a aba se não existirem, adiciona cabeçalhos se a aba estiver vazia.
    """
    # Define a ordem e os cabeçalhos para o Excel
    colunas_excel = [
        "Data do Exame", "Nome", "ID", "Idade", "Sexo", "Altura (cm)", "Peso Atual (kg)", 
        "Água Corporal Total (L)", "Proteína Atual (kg)", "Minerais Atual (kg)",
        "Massa de Gordura Atual (kg)", "Massa Muscular Esquelética Atual (kg)", 
        "IMC", "PGC (%)", "Pontuação InBody",
        "Peso Histórico Recente (kg)", "Massa Muscular Histórico Recente (kg)", "PGC Histórico Recente (%)", "Proteína Histórico Recente (kg)"
    ]
    
    # Mapeia as chaves do dicionário 'dados' para as colunas do Excel
    # Ajuste aqui para mapear as chaves que suas regex estão gerando
    mapeamento_chaves = {
        "Data do Exame": "Data_Hora_Exame",
        "Nome": "Nome",
        "ID": "ID",
        "Idade": "Idade",
        "Sexo": "Sexo",
        "Altura (cm)": "Altura",
        "Peso Atual (kg)": "Peso_Atual",
        "Água Corporal Total (L)": "Agua_Corporal_Total_Atual",
        "Proteína Atual (kg)": "Proteina_Atual",
        "Minerais Atual (kg)": "Minerais_Atual",
        "Massa de Gordura Atual (kg)": "Massa_Gordura_Atual",
        "Massa Muscular Esquelética Atual (kg)": "Massa_Muscular_Esqueletica_Atual",
        "IMC": "IMC_Obesidade",
        "PGC (%)": "PGC_Obesidade",
        "Pontuação InBody": "Pontuacao_Inbody",
        "Peso Histórico Recente (kg)": "Peso_Historico_Recente",
        "Massa Muscular Histórico Recente (kg)": "Massa_Muscular_Historico_Recente",
        "PGC Histórico Recente (%)": "PGC_Historico_Recente",
        "Proteína Histórico Recente (kg)": "Proteina_Historico_Recente"
    }

    try:
        # Tenta carregar o arquivo existente
        if os.path.exists(caminho_excel):
            workbook = load_workbook(caminho_excel)
        else:
            # Se não existir, cria um novo
            workbook = Workbook()
            # Remove a planilha padrão criada (geralmente "Sheet")
            if 'Sheet' in workbook.sheetnames:
                workbook.remove(workbook['Sheet'])

        # Tenta obter a aba existente ou cria uma nova
        if nome_aba_excel in workbook.sheetnames:
            sheet = workbook[nome_aba_excel]
        else:
            sheet = workbook.create_sheet(nome_aba_excel)
            
        # Adiciona cabeçalhos se a aba estiver vazia
        if sheet.max_row == 0 or sheet.cell(row=1, column=1).value is None:
            sheet.append(colunas_excel)
            print(f"Cabeçalhos adicionados à aba '{nome_aba_excel}'.")

        # Prepara a linha de dados para inserção
        linha_para_inserir = []
        for coluna_nome in colunas_excel:
            chave_interna = mapeamento_chaves.get(coluna_nome)
            if chave_interna:
                valor = dados.get(chave_interna, "")
                
                # Tenta converter para float se for um número, para melhor formatação no Excel
                try:
                    # Remove caracteres não numéricos para a conversão, mas mantém o ponto decimal
                    # Remove o 'kg' ou 'cm' ou '%' antes de converter
                    valor_limpo_para_num = re.sub(r'[^\d.]+', '', str(valor).replace(',', '.'))
                    linha_para_inserir.append(float(valor_limpo_para_num))
                except (ValueError, TypeError):
                    linha_para_inserir.append(valor) # Mantém como string se não for número

            else:
                linha_para_inserir.append("") # Campo não mapeado

        sheet.append(linha_para_inserir)
        
        # Ajusta a largura das colunas automaticamente (opcional)
        for col_idx, col_name in enumerate(colunas_excel, 1):
            max_length = 0
            for row in sheet.iter_rows(min_row=1, min_col=col_idx, max_col=col_idx):
                for cell in row:
                    try:
                        if cell.value is not None:
                            max_length = max(max_length, len(str(cell.value)))
                    except TypeError: # Tratar caso de célula ter tipo não-string
                        pass
            adjusted_width = (max_length + 2) * 1.2 # Fator de ajuste para legibilidade
            sheet.column_dimensions[get_column_letter(col_idx)].width = adjusted_width

        workbook.save(caminho_excel)
        print(f"✅ Dados enviados ao arquivo Excel: '{caminho_excel}' na aba '{nome_aba_excel}' com sucesso.")

    except Exception as e:
        print(f"❌ Ocorreu um erro ao salvar no Excel: {e}")

# --- Execução Principal ---
print(f"Iniciando extração de dados da imagem: {caminho_imagem}")
dados_extraidos = extrair_dados_inbody_da_imagem(caminho_imagem)

if not dados_extraidos:
    print("❌ Falha ao extrair dados da imagem. Verifique o caminho da imagem e a saída do OCR.")
else:
    print("\n--- Dados Extraídos para Envio ---")
    for k, v in dados_extraidos.items():
        print(f"{k}: {v}")
    print("------------------------------\n")
    
    # --- Salvar os dados extraídos no arquivo Excel local ---
    salvar_no_excel(dados_extraidos, caminho_excel, nome_aba_excel)